# 법령 API v2 수집

법제처 API의 `eflaw`, `prec`, `expc` 응답을 OC만 제거한 append-only JSONL로 보존하고, 원문 검수 Markdown과 전처리 전 Document JSONL을 생성한다. 이 Notebook은 토큰 chunking, 임베딩, 벡터 DB 적재를 수행하지 않는다.

기본값은 `sample_*` run이다. 같은 `RUN_ID` 재실행 시 이미 저장된 검색 페이지와 상세 ID는 API를 다시 호출하지 않는다. 새 수집은 반드시 새 `RUN_ID`로 실행한다.

In [1]:
from __future__ import annotations

import hashlib
import html
import json
import math
import os
import re
import time
from collections import defaultdict
from datetime import datetime, timezone
from functools import lru_cache
from pathlib import Path
from typing import Any, Iterable, Iterator, Literal
from urllib.parse import parse_qs, urljoin, urlsplit

import requests
from dotenv import load_dotenv
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / '.git').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('프로젝트 루트를 찾지 못했습니다.')


PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / '.env')
LAW_API_OC = os.getenv('LAW_API_OC', '').strip()
if not LAW_API_OC or LAW_API_OC == 'your-oc-here':
    raise RuntimeError('.env에 LAW_API_OC를 설정하세요.')

SAMPLE_MODE = os.getenv('LEGAL_API_SAMPLE_MODE', 'true').strip().lower() != 'false'
RUN_KIND = 'sample' if SAMPLE_MODE else 'full'
RUN_ID = os.getenv('LEGAL_API_RUN_ID') or f"{RUN_KIND}_{datetime.now():%Y%m%d_%H%M%S}"
REQUEST_TIMEOUT = 60
REQUEST_MAX_ATTEMPTS = 4
REQUEST_INTERVAL_SECONDS = 0.22  # 초당 최대 5회보다 보수적으로 제한
PRECEDENT_MAX_PER_QUERY: int | None = 300
PRECEDENT_SORT = 'ddes'
SAMPLE_EFLAW_QUERY = '공인중개사법 시행규칙'  # 계층·별표·이미지 참조 검증용
JSONL_MAX_BYTES = 50 * 1024 * 1024
MARKDOWN_MAX_DOCUMENTS = 200

DATA_ROOT = PROJECT_ROOT / 'data' / 'legal_api_v2'
RAW_ROOT = DATA_ROOT / '01_raw_json' / RUN_ID
RAW_MD_ROOT = DATA_ROOT / '02_raw_md' / RUN_ID
DOCUMENT_ROOT = DATA_ROOT / '03_document_json' / RUN_ID
MANIFEST_PATH = RAW_ROOT / 'collection_manifest.json'

print(f'LAW_API_OC 설정 여부: {bool(LAW_API_OC)}')
print(f'RUN_ID: {RUN_ID} | SAMPLE_MODE: {SAMPLE_MODE}')
print(f'저장 경로: {DATA_ROOT}')

LAW_API_OC 설정 여부: True
RUN_ID: full_20260719_163719 | SAMPLE_MODE: False
저장 경로: D:\project\SKN30-3rd-4Team-dev\data\legal_api_v2


In [2]:
EFLAW_QUERIES = [
    '주택임대차보호법', '주택임대차보호법 시행령', '민법', '부동산등기법',
    '공인중개사법', '민사집행법', '전세사기피해자 지원 및 주거안정에 관한 특별법',
    '부동산 거래신고 등에 관한 법률', '주민등록법', '상가건물 임대차보호법',
    '부동산등기규칙', '공인중개사법 시행규칙', '민간임대주택에 관한 특별법',
    '국세징수법', '집합건물의 소유 및 관리에 관한 법률', '주택도시기금법',
]

PREC_QUERY_CONFIG = [
    ('임대차보증금', '보증금권리'), ('임대차보증금 반환', '보증금권리'),
    ('임차인 대항력', '보증금권리'), ('임차인 우선변제권', '보증금권리'),
    ('소액임차인 최우선변제', '보증금권리'), ('임대차 확정일자', '보증금권리'),
    ('임차권등기명령', '보증금권리'), ('임차인 전입신고', '보증금권리'),
    ('계약갱신청구권', '갱신종료'), ('임대차 묵시적 갱신', '갱신종료'),
    ('임대차 갱신거절', '갱신종료'), ('임대차 해지', '갱신종료'),
    ('임차인 차임 연체', '갱신종료'), ('임대차 차임 증감', '갱신종료'),
    ('전세사기', '전세사기'), ('임대차 가장임대차', '전세사기'),
    ('임대차 무권대리', '전세사기'), ('임대차 이중계약', '전세사기'),
    ('임대차보증금 사해행위', '전세사기'), ('임대차보증금 명의신탁', '전세사기'),
    ('임차인 임의경매', '경매배당'), ('임차인 강제경매', '경매배당'),
    ('임차인 배당요구', '경매배당'), ('임차인 배당이의', '경매배당'),
    ('임차인 인도명령', '경매배당'), ('임대차 건물명도', '경매배당'),
    ('임대인 수선의무', '수선원상회복'), ('임대차 원상회복', '수선원상회복'),
    ('임대차 누수', '수선원상회복'), ('임대차 하자', '수선원상회복'),
    ('임대차 통상의 손모', '수선원상회복'), ('공인중개사 책임', '중개'),
    ('중개대상물 확인설명', '중개'), ('부동산 중개보수', '중개'),
]

EXPC_QUERY_CONFIG = [
    ('주택임대차보호법', '보증금권리'), ('임대차보증금 우선변제', '보증금권리'),
    ('대항력', '보증금권리'), ('확정일자', '보증금권리'),
    ('임차권등기명령', '보증금권리'), ('소액임차인 최우선변제', '보증금권리'),
    ('전입신고', '보증금권리'), ('계약갱신청구권', '갱신종료'),
    ('묵시적 갱신', '갱신종료'), ('차임 증액', '갱신종료'),
    ('상가건물 임대차', '상가임대차'),
]

TARGET_CONFIG = {
    'eflaw': {'root': 'LawSearch', 'items': 'law', 'id': '법령ID'},
    'prec': {'root': 'PrecSearch', 'items': 'prec', 'id': '판례일련번호'},
    'expc': {'root': 'Expc', 'items': 'expc', 'id': '법령해석례일련번호'},
}

## OC 제거, JSONL append-only 저장, 해시

In [3]:
_OC_QUERY_RE = re.compile(
    r'(?i)([?&](?:amp;)?)oc[=][^&#\s\x22\x27<>]*([&](?:amp;)?|(?=[#\s\x22\x27<>]|$))'
)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def canonical_json_bytes(value: Any) -> bytes:
    return json.dumps(value, ensure_ascii=False, separators=(',', ':')).encode('utf-8')


def _redact_oc_in_string(value: str) -> str:
    def replace(match: re.Match[str]) -> str:
        left, right = match.group(1), match.group(2)
        if not right:
            return ''
        return '?' if left.startswith('?') else right

    previous = None
    while previous != value:
        previous = value
        value = _OC_QUERY_RE.sub(replace, value)
    return value


def sanitize_oc(value: Any) -> Any:
    if isinstance(value, dict):
        return {
            key: ('__REDACTED__' if str(key).lower() == 'oc' else sanitize_oc(item))
            for key, item in value.items()
        }
    if isinstance(value, list):
        return [sanitize_oc(item) for item in value]
    if isinstance(value, str):
        return _redact_oc_in_string(value)
    return value


def contains_oc_assignment(value: Any) -> bool:
    needle = 'OC' + '='
    return needle.lower() in json.dumps(value, ensure_ascii=False).lower()


class JsonlStore:
    def __init__(self, path: Path, max_bytes: int = JSONL_MAX_BYTES) -> None:
        self.path = path
        self.max_bytes = max_bytes
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self._records_cache: list[dict[str, Any]] | None = None

    def paths(self) -> list[Path]:
        parts = sorted(self.path.parent.glob(f'{self.path.stem}.part*{self.path.suffix}'))
        return parts or ([self.path] if self.path.exists() else [])

    def records(self) -> Iterator[dict[str, Any]]:
        if self._records_cache is None:
            self._records_cache = []
            for path in self.paths():
                with path.open(encoding='utf-8') as handle:
                    for line_no, line in enumerate(handle, 1):
                        if line.strip():
                            try:
                                self._records_cache.append(json.loads(line))
                            except json.JSONDecodeError as exc:
                                raise ValueError(f'{path}:{line_no} JSON 오류') from exc
        yield from self._records_cache

    def find(self, **keys: Any) -> dict[str, Any] | None:
        def value_at(row: dict[str, Any], dotted_key: str) -> Any:
            value: Any = row
            for part in dotted_key.split('.'):
                value = value.get(part) if isinstance(value, dict) else None
            return value
        return next((row for row in self.records() if all(value_at(row, k) == v for k, v in keys.items())), None)

    def append(self, record: dict[str, Any], unique_keys: tuple[str, ...]) -> bool:
        def value_at(row: dict[str, Any], dotted_key: str) -> Any:
            value: Any = row
            for part in dotted_key.split('.'):
                value = value.get(part) if isinstance(value, dict) else None
            return value
        wanted = {key: value_at(record, key) for key in unique_keys}
        if self.find(**wanted) is not None:
            return False
        encoded = canonical_json_bytes(record) + b'\n'
        target = self._append_path(len(encoded))
        with target.open('ab') as handle:
            handle.write(encoded)
        if self._records_cache is not None:
            self._records_cache.append(record)
        return True

    def _append_path(self, incoming_bytes: int) -> Path:
        paths = self.paths()
        if not paths:
            return self.path
        current = paths[-1]
        if current.stat().st_size + incoming_bytes <= self.max_bytes:
            return current
        if current == self.path:
            first = self.path.with_name(f'{self.path.stem}.part01{self.path.suffix}')
            self.path.replace(first)
            part_no = 2
        else:
            match = re.search(r'part(\d+)$', current.stem)
            part_no = int(match.group(1)) + 1 if match else len(paths) + 1
        return self.path.with_name(f'{self.path.stem}.part{part_no:02d}{self.path.suffix}')


def make_envelope(*, source_id: str, query: str, page: int | None, raw_bytes: bytes, response: dict[str, Any]) -> dict[str, Any]:
    sanitized = sanitize_oc(response)
    if contains_oc_assignment(sanitized):
        raise ValueError(f'OC 제거 검증 실패: source_id={source_id!r}, query={query!r}, page={page!r}')
    return {
        'source_id': source_id, 'query': query, 'page': page, 'fetched_at': utc_now(),
        'sha256_original': sha256_bytes(raw_bytes),
        'sha256_sanitized': sha256_bytes(canonical_json_bytes(sanitized)),
        'response': sanitized,
    }

## HTTP 요청, resultCode 검증, 검색 수집

In [4]:
SEARCH_URL = 'https://www.law.go.kr/DRF/lawSearch.do'
DETAIL_URL = 'https://www.law.go.kr/DRF/lawService.do'


class ApiResponseError(RuntimeError):
    pass


class LawApiClient:
    def __init__(self, oc: str) -> None:
        self.oc = oc
        retry = Retry(total=3, backoff_factor=1.5, status_forcelist=[429, 500, 502, 503, 504], allowed_methods=['GET'])
        self.session = requests.Session()
        self.session.mount('https://', HTTPAdapter(max_retries=retry))
        self._last_request_at = 0.0
        self.failures: list[dict[str, Any]] = []

    def get(self, target: str, service: Literal['search', 'detail'], params: dict[str, Any]) -> tuple[bytes, dict[str, Any]]:
        url = SEARCH_URL if service == 'search' else DETAIL_URL
        request_params = {'OC': self.oc, 'type': 'JSON', 'target': target, **params}
        last_error: Exception | None = None
        for attempt in range(1, REQUEST_MAX_ATTEMPTS + 1):
            wait = REQUEST_INTERVAL_SECONDS - (time.monotonic() - self._last_request_at)
            if wait > 0:
                time.sleep(wait)
            try:
                response = self.session.get(url, params=request_params, timeout=REQUEST_TIMEOUT)
                self._last_request_at = time.monotonic()
                response.raise_for_status()
                raw_bytes = response.content
                payload = response.json()
                if service == 'search':
                    root = search_root(payload, target)
                    result_code = str(root.get('resultCode', payload.get('resultCode', '00')))
                    if result_code != '00':
                        message = root.get('resultMsg', payload.get('resultMsg', ''))
                        raise ApiResponseError(f'{target} 검색 오류 {result_code}: {message}')
                return raw_bytes, payload
            except (requests.RequestException, ValueError, ApiResponseError) as exc:
                last_error = exc
                self.failures.append({'at': utc_now(), 'target': target, 'service': service, 'attempt': attempt, 'type': type(exc).__name__, 'message': str(exc)})
                if attempt < REQUEST_MAX_ATTEMPTS:
                    time.sleep(1.5 * attempt)
        raise ApiResponseError(f'{target}/{service} 요청이 {REQUEST_MAX_ATTEMPTS}회 실패했습니다.') from last_error


def search_root(response: dict[str, Any], target: str) -> dict[str, Any]:
    root = response.get(TARGET_CONFIG[target]['root'], {})
    return root if isinstance(root, dict) else {}


def search_items(response: dict[str, Any], target: str) -> list[dict[str, Any]]:
    items = search_root(response, target).get(TARGET_CONFIG[target]['items'], [])
    if isinstance(items, dict):
        items = [items]
    return [item for item in items if isinstance(item, dict)] if isinstance(items, list) else []


@lru_cache(maxsize=None)
def raw_search_store(target: str) -> JsonlStore:
    return JsonlStore(RAW_ROOT / target / f'{target}_search.jsonl')


def load_or_fetch_search_page(client: LawApiClient, target: str, query: str, page: int, params: dict[str, Any]) -> dict[str, Any]:
    store = raw_search_store(target)
    existing = store.find(query=query, page=page)
    if existing is not None:
        return existing['response']
    raw_bytes, response = client.get(target, 'search', {**params, 'page': page})
    store.append(make_envelope(source_id='', query=query, page=page, raw_bytes=raw_bytes, response=response), ('query', 'page'))
    return sanitize_oc(response)


def collect_query(client: LawApiClient, target: str, query: str, category: str, *, search: int, max_items: int | None, extra: dict[str, Any] | None = None) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    display = 100
    params = {'query': query, 'search': search, 'display': display, **(extra or {})}
    first = load_or_fetch_search_page(client, target, query, 1, params)
    total_count = int(search_root(first, target).get('totalCnt', 0) or 0)
    requested_count = min(total_count, max_items) if max_items is not None else total_count
    page_count = math.ceil(requested_count / display) if requested_count else 1
    items = search_items(first, target)
    for page in range(2, page_count + 1):
        items.extend(search_items(load_or_fetch_search_page(client, target, query, page, params), target))
    items = items[:requested_count]
    stats = {
        'query': query, 'category': category, 'search_scope': 'body' if search == 2 else 'name',
        'sort': extra.get('sort') if extra else None, 'max_per_query': max_items,
        'page_count': page_count,
        'total_count': total_count, 'collected_count': len(items),
        'truncated_count': max(0, total_count - len(items)),
    }
    return items, stats


def add_candidate(candidates: dict[str, dict[str, Any]], target: str, item: dict[str, Any], query: str, category: str) -> None:
    source_id = str(item.get(TARGET_CONFIG[target]['id'], '')).strip()
    if not source_id:
        return
    record = candidates.setdefault(source_id, {'item': item, 'queries': [], 'categories': []})
    if query not in record['queries']:
        record['queries'].append(query)
    if category not in record['categories']:
        record['categories'].append(category)


def collect_candidates(client: LawApiClient, target: str, query_config: list[tuple[str, str]], manifest: dict[str, Any]) -> dict[str, dict[str, Any]]:
    if SAMPLE_MODE and target == 'eflaw':
        selected = [spec for spec in query_config if spec[0] == SAMPLE_EFLAW_QUERY]
        if len(selected) != 1:
            raise ValueError(f'샘플 법령 검색어를 정확히 하나 찾지 못했습니다: {SAMPLE_EFLAW_QUERY}')
    else:
        selected = query_config[:2] if SAMPLE_MODE else query_config
    candidates: dict[str, dict[str, Any]] = {}
    manifest['query_stats'][target] = []
    for query, category in selected:
        if target == 'eflaw':
            items, stats = collect_query(client, target, query, category, search=1, max_items=None, extra={'nw': 3})
            items = [item for item in items if str(item.get('법령명한글', '')).strip() == query and item.get('현행연혁코드') == '현행']
            stats['exact_current_count'] = len(items)
        elif target == 'prec':
            limit = 5 if SAMPLE_MODE else PRECEDENT_MAX_PER_QUERY
            items, stats = collect_query(client, target, query, category, search=2, max_items=limit, extra={'sort': PRECEDENT_SORT})
            stats['selection_policy'] = 'latest_per_query'
            stats['relevance_evaluated_after_collection'] = True
        else:
            items, stats = collect_query(client, target, query, category, search=2, max_items=5 if SAMPLE_MODE else None)
        manifest['query_stats'][target].append(stats)
        for item in items:
            add_candidate(candidates, target, item, query, category)
        print(f'[{target}] {query}: 검색 {stats["total_count"]} / 수집 {len(items)}')
    return candidates

## 상세 응답 원형 보존과 resume

In [5]:
def safe_name(value: str) -> str:
    value = re.sub(r'[\\/:*?"<>|]+', '_', value.strip())
    return re.sub(r'\s+', '_', value).strip('_') or 'unknown'


@lru_cache(maxsize=None)
def raw_detail_store(target: str, category: str = '') -> JsonlStore:
    name = f'prec_detail_{safe_name(category)}.jsonl' if target == 'prec' else f'{target}_detail.jsonl'
    return JsonlStore(RAW_ROOT / target / name)


def detail_stores(target: str) -> list[JsonlStore]:
    if target != 'prec':
        return [raw_detail_store(target)]
    bases: dict[str, Path] = {}
    for path in (RAW_ROOT / 'prec').glob('prec_detail_*.jsonl'):
        stem = re.sub(r'\.part\d+$', '', path.stem)
        bases[stem] = path.with_name(stem + '.jsonl')
    return [JsonlStore(path) for path in sorted(bases.values())]


def find_saved_detail(target: str, source_id: str) -> dict[str, Any] | None:
    for store in detail_stores(target):
        found = store.find(source_id=source_id)
        if found is not None:
            return found
    return None


def detail_path_index(target: str) -> dict[str, str]:
    index: dict[str, str] = {}
    for store in detail_stores(target):
        for path in store.paths():
            with path.open(encoding='utf-8') as handle:
                for line in handle:
                    if not line.strip():
                        continue
                    row = json.loads(line)
                    if row.get('source_id'):
                        index[row['source_id']] = str(path.relative_to(PROJECT_ROOT)).replace('\\', '/')
    return index


def detail_payload(response: dict[str, Any], target: str) -> Any:
    key = {'eflaw': '법령', 'prec': 'PrecService', 'expc': 'ExpcService'}[target]
    return response.get(key)


def is_invalid_detail(response: dict[str, Any], target: str) -> bool:
    payload = detail_payload(response, target)
    if not payload:
        return True
    text = json.dumps(payload, ensure_ascii=False)
    return '일치하는' in text and '없습니다' in text


def collect_details(client: LawApiClient, target: str, candidates: dict[str, dict[str, Any]], manifest: dict[str, Any]) -> list[dict[str, Any]]:
    envelopes: list[dict[str, Any]] = []
    saved_details = {row['source_id']: row for store in detail_stores(target) for row in store.records() if row.get('source_id')}
    invalid_ids: list[str] = []
    for index, (source_id, provenance) in enumerate(candidates.items(), 1):
        envelope = saved_details.get(source_id)
        if envelope is None:
            raw_bytes, response = client.get(target, 'detail', {'ID': source_id})
            envelope = make_envelope(
                source_id=source_id, query=provenance['queries'][0], page=None,
                raw_bytes=raw_bytes, response=response,
            )
            category = provenance['categories'][0] if provenance['categories'] else '기타'
            raw_detail_store(target, category).append(envelope, ('source_id',))
            saved_details[source_id] = envelope
        envelopes.append(envelope)
        if is_invalid_detail(envelope['response'], target):
            invalid_ids.append(source_id)
        if index % 20 == 0 or index == len(candidates):
            print(f'[{target}] 상세 {index}/{len(candidates)} | 오류·빈 본문 {len(invalid_ids)}')
    source_paths = detail_path_index(target)
    for envelope in envelopes:
        envelope['_source_file'] = source_paths.get(envelope['source_id'], '')
    manifest['invalid_details'][target] = invalid_ids
    manifest['detail_counts'][target] = {
        'requested': len(candidates), 'stored': len(envelopes),
        'valid': len(envelopes) - len(invalid_ids), 'invalid': len(invalid_ids),
        'invalid_ratio': (len(invalid_ids) / len(envelopes)) if envelopes else 0.0,
    }
    manifest['provenance'][target] = {
        source_id: {'queries': row['queries'], 'categories': row['categories']}
        for source_id, row in candidates.items()
    }
    return envelopes

## 전처리 전 구조 Document와 이미지 참조

In [6]:
_IMG_RE = re.compile(r'<img\b[^>]*>', re.IGNORECASE)
_ATTR_RE = re.compile(r'(src|alt)\s*=\s*[\x22\x27]([^\x22\x27]*)[\x22\x27]', re.IGNORECASE)
_HEADING_RE = re.compile(r'^\s*(제[^\s]+(?:편|장|절|관))\s*(.*)$')


def as_list(value: Any) -> list[Any]:
    if value is None:
        return []
    return value if isinstance(value, list) else [value]


def flatten_raw(value: Any) -> str:
    if value is None:
        return ''
    if isinstance(value, dict):
        return '\n'.join(part for item in value.values() if (part := flatten_raw(item)))
    if isinstance(value, list):
        return '\n'.join(part for item in value if (part := flatten_raw(item)))
    return str(value)


def content_value(value: Any) -> str:
    if isinstance(value, dict) and value.get('content') not in (None, ''):
        return str(value['content'])
    return flatten_raw(value)


def walk_values(value: Any, key: str = '') -> Iterator[tuple[str, Any]]:
    if isinstance(value, dict):
        for child_key, child in value.items():
            yield child_key, child
            yield from walk_values(child, child_key)
    elif isinstance(value, list):
        for child in value:
            if isinstance(child, (dict, list)):
                yield from walk_values(child, key)
            else:
                yield key, child


def walk_selected_fields(value: Any, field_names: set[str]) -> Iterator[tuple[str, Any]]:
    if isinstance(value, dict):
        for key, child in value.items():
            if key in field_names:
                yield key, child
            else:
                yield from walk_selected_fields(child, field_names)
    elif isinstance(value, list):
        for child in value:
            yield from walk_selected_fields(child, field_names)


def collect_text_fields(value: Any, field_names: set[str]) -> str:
    return '\n'.join(
        text for _, child in walk_selected_fields(value, field_names)
        if (text := flatten_raw(child))
    )


def normalize_url(url: str) -> str:
    return urljoin('https://www.law.go.kr', _redact_oc_in_string(html.unescape(url)))


def image_reference(url: str, alt: str = '', source_section: str = '') -> dict[str, Any]:
    normalized = normalize_url(url)
    query = parse_qs(urlsplit(normalized).query)
    image_id = next(iter(query.get('flSeq', [])), '') or alt.removeprefix('img')
    return {'image_id': image_id, 'image_url': normalized, 'alt': alt, 'source_section': source_section}


def extract_images(value: Any) -> list[dict[str, Any]]:
    images: list[dict[str, Any]] = []
    seen: set[tuple[str, str]] = set()
    for key, child in walk_values(value):
        if isinstance(child, str):
            for tag in _IMG_RE.findall(child):
                attrs = {name.lower(): val for name, val in _ATTR_RE.findall(tag)}
                if attrs.get('src'):
                    ref = image_reference(attrs['src'], attrs.get('alt', ''), key)
                    marker = (ref['image_url'], key)
                    if marker not in seen:
                        seen.add(marker); images.append(ref)
            if '이미지' in key and ('flDownload' in child or child.startswith('/LSW/')):
                ref = image_reference(child, source_section=key)
                marker = (ref['image_url'], key)
                if marker not in seen:
                    seen.add(marker); images.append(ref)
    for order, ref in enumerate(images, 1):
        ref['image_order'] = order
    return images


def raw_source_path(target: str, envelope: dict[str, Any]) -> str:
    return str(envelope.get('_source_file', ''))


def common_metadata(target: str, envelope: dict[str, Any], provenance: dict[str, Any], payload: dict[str, Any]) -> dict[str, Any]:
    source_id = envelope['source_id']
    source_type = {'eflaw': 'statute', 'prec': 'precedent', 'expc': 'interpretation'}[target]
    title = payload.get('법령명_한글') or payload.get('법령명한글') or payload.get('사건명') or payload.get('안건명') or source_id
    court = payload.get('법원명', '')
    return {
        'source_type': source_type, 'source_id': source_id, 'parent_id': source_id,
        'doc_title': title, 'source_org': payload.get('소관부처명') or content_value(payload.get('소관부처', '')) or court or payload.get('해석기관명', ''),
        'doc_year': str(payload.get('선고일자') or payload.get('해석일자') or payload.get('공포일자') or '')[:4],
        'authority': 'binding' if target == 'eflaw' or (target == 'prec' and court == '대법원') else 'persuasive',
        'stage': 'raw_document', 'issue': provenance.get('categories', [''])[0] if provenance.get('categories') else '',
        'source_file': raw_source_path(target, envelope), 'collection_run_id': RUN_ID,
        'collection_queries': provenance.get('queries', []), 'collection_categories': provenance.get('categories', []),
        'raw_sha256_original': envelope['sha256_original'], 'raw_sha256_sanitized': envelope['sha256_sanitized'],
    }


def document(page_content: str, metadata: dict[str, Any], section: str, record_id: str, source_value: Any) -> dict[str, Any]:
    images = extract_images(source_value)
    return {
        'page_content': page_content,
        'metadata': {**metadata, 'record_id': record_id, 'section': section, 'content_type': 'text',
                     'has_image': bool(images), 'image_count': len(images), 'images': images},
    }

In [7]:
def eflaw_documents(envelope: dict[str, Any], provenance: dict[str, Any]) -> list[dict[str, Any]]:
    law = detail_payload(envelope['response'], 'eflaw')
    if not isinstance(law, dict):
        return []
    basic = law.get('기본정보', {}) if isinstance(law.get('기본정보'), dict) else {}
    meta = common_metadata('eflaw', envelope, provenance, basic)
    meta.update({
        'law_id': envelope['source_id'], 'law_name': meta['doc_title'], 'law_type': content_value(basic.get('법종구분', '')),
        'effective_date': basic.get('시행일자', ''), 'promulgation_date': basic.get('공포일자', ''),
        'promulgation_no': basic.get('공포번호', ''), 'revision_type': basic.get('제개정구분', ''), 'ministry': content_value(basic.get('소관부처', '')),
    })
    docs: list[dict[str, Any]] = []
    hierarchy = {'part': '', 'chapter': '', 'division': '', 'subdivision': ''}
    level_map = {'편': 'part', '장': 'chapter', '절': 'division', '관': 'subdivision'}
    units = law.get('조문', {}).get('조문단위', []) if isinstance(law.get('조문'), dict) else []
    for index, unit in enumerate(as_list(units), 1):
        if not isinstance(unit, dict):
            continue
        content = collect_text_fields(unit, {'조문내용', '항내용', '호내용', '목내용'})
        key = str(unit.get('조문키') or f'{index:07d}')
        if unit.get('조문여부') == '전문':
            heading_text = str(unit.get('조문내용', content)).strip()
            match = _HEADING_RE.match(heading_text)
            if match:
                level = level_map[match.group(1)[-1]]
                hierarchy[level] = heading_text
                levels = list(level_map.values()); position = levels.index(level)
                for lower in levels[position + 1:]: hierarchy[lower] = ''
            record_meta = {**meta, **hierarchy, 'structure_path': [v for v in hierarchy.values() if v]}
            docs.append(document(content, record_meta, 'heading', f'{envelope["source_id"]}:heading:{key}', unit))
            continue
        article = str(unit.get('조문내용', '')).strip()
        article_no = str(unit.get('조문번호', '')).strip()
        notes = flatten_raw(unit.get('조문참고자료', ''))
        record_meta = {
            **meta, **hierarchy, 'structure_path': [v for v in hierarchy.values() if v],
            'article': article.split('(', 1)[0].strip(), 'article_no': article_no,
            'article_title': unit.get('조문제목', ''), 'article_key': key,
            'is_deleted': '삭제' in article, 'revision_notes': notes,
        }
        docs.append(document(content, record_meta, 'article', f'{envelope["source_id"]}:article:{key}', unit))
    for section, container_key, unit_key, id_key in [
        ('supplement', '부칙', '부칙단위', '부칙키'), ('annex', '별표', '별표단위', '별표키')
    ]:
        container = law.get(container_key, {})
        values = container.get(unit_key, []) if isinstance(container, dict) else container
        for index, unit in enumerate(as_list(values), 1):
            if not isinstance(unit, dict):
                continue
            key = str(unit.get(id_key) or f'{index:05d}')
            fields = {'부칙내용'} if section == 'supplement' else {'별표제목', '별표제목문자열', '별표내용'}
            docs.append(document(collect_text_fields(unit, fields), meta, section, f'{envelope["source_id"]}:{section}:{key}', unit))
    return docs


def prec_documents(envelope: dict[str, Any], provenance: dict[str, Any]) -> list[dict[str, Any]]:
    payload = detail_payload(envelope['response'], 'prec')
    if not isinstance(payload, dict):
        return []
    meta = common_metadata('prec', envelope, provenance, payload)
    meta.update({
        'precedent_id': envelope['source_id'], 'court': payload.get('법원명', ''), 'case_no': payload.get('사건번호', ''),
        'decision_date': payload.get('선고일자', ''), 'decision_type': payload.get('선고', ''),
        'case_type': payload.get('사건종류명', ''), 'judgment_type': payload.get('판결유형', ''),
        'reference_laws': flatten_raw(payload.get('참조조문', '')), 'reference_cases': flatten_raw(payload.get('참조판례', '')),
        'selection_policy': 'latest_per_query', 'sort': PRECEDENT_SORT, 'max_per_query': PRECEDENT_MAX_PER_QUERY,
    })
    result = []
    for section, field in [('holding', '판시사항'), ('summary', '판결요지'), ('body', '판례내용')]:
        value = payload.get(field)
        if value not in (None, ''):
            result.append(document(flatten_raw(value), meta, section, f'{envelope["source_id"]}:{section}', value))
    return result


def expc_documents(envelope: dict[str, Any], provenance: dict[str, Any]) -> list[dict[str, Any]]:
    payload = detail_payload(envelope['response'], 'expc')
    if not isinstance(payload, dict):
        return []
    meta = common_metadata('expc', envelope, provenance, payload)
    meta.update({
        'interpretation_id': envelope['source_id'], 'case_no': payload.get('안건번호', ''),
        'interpreting_agency': payload.get('해석기관명', ''), 'requesting_agency': payload.get('질의기관명', ''),
        'decision_date': payload.get('해석일자', ''), 'registration_date': payload.get('등록일시', ''),
        'law_name': payload.get('법령명', ''), 'article': payload.get('조문', ''),
    })
    result = []
    for section, field in [('question', '질의요지'), ('answer', '회답'), ('reason', '이유')]:
        value = payload.get(field)
        if value not in (None, ''):
            result.append(document(flatten_raw(value), meta, section, f'{envelope["source_id"]}:{section}', value))
    return result


def write_documents(target: str, envelopes: list[dict[str, Any]], manifest: dict[str, Any]) -> int:
    stores: dict[str, JsonlStore] = {}
    provenance_map = manifest['provenance'][target]
    for envelope in envelopes:
        if is_invalid_detail(envelope['response'], target):
            continue
        provenance = provenance_map[envelope['source_id']]
        docs = {'eflaw': eflaw_documents, 'prec': prec_documents, 'expc': expc_documents}[target](envelope, provenance)
        categories = provenance.get('categories') or ['기타']
        output_names = ['eflaw'] if target == 'eflaw' else ([f'prec_{safe_name(categories[0])}'] if target == 'prec' else ['expc'])
        for output_name in output_names:
            store = stores.setdefault(output_name, JsonlStore(DOCUMENT_ROOT / f'{output_name}.jsonl'))
            for doc in docs:
                store.append(doc, ('metadata.record_id',))
    return sum(1 for store in stores.values() for _ in store.records())

## 원문 검수 Markdown

In [8]:
def markdown_value(value: Any) -> str:
    text = flatten_raw(value)
    def replace_image(match: re.Match[str]) -> str:
        attrs = {name.lower(): val for name, val in _ATTR_RE.findall(match.group(0))}
        if not attrs.get('src'):
            return match.group(0)
        ref = image_reference(attrs['src'], attrs.get('alt', ''))
        label = ref['image_id'] or ref['alt'] or 'image'
        return f'![{label}]({ref["image_url"]})'
    return _IMG_RE.sub(replace_image, text)


def raw_markdown(target: str, envelope: dict[str, Any]) -> str:
    payload = detail_payload(envelope['response'], target)
    if not isinstance(payload, dict):
        return ''
    basic = payload.get('기본정보', {}) if target == 'eflaw' and isinstance(payload.get('기본정보'), dict) else payload
    title = basic.get('법령명_한글') or basic.get('법령명한글') or payload.get('사건명') or payload.get('안건명') or envelope['source_id']
    wanted = {
        'eflaw': {'조문내용', '항내용', '호내용', '목내용', '조문참고자료', '부칙내용', '별표제목', '별표내용', '개정문내용', '제개정이유내용'},
        'prec': {'판시사항', '판결요지', '참조조문', '참조판례', '판례내용'},
        'expc': {'질의요지', '회답', '이유'},
    }[target]
    sections = [f'# {title}', f'<!-- source_id: {envelope["source_id"]} -->']
    for key, value in walk_selected_fields(payload, wanted):
        if value not in (None, '', [], {}):
            sections.extend([f'## {key}', markdown_value(value)])
    images = extract_images(payload)
    if images:
        sections.append('## 이미지 참조')
        sections.extend(f'- [{ref["image_id"] or ref["alt"] or "image"}]({ref["image_url"]}) — {ref["source_section"]}' for ref in images)
    return '\n\n'.join(sections).strip() + '\n'


def write_raw_markdown(target: str, envelopes: list[dict[str, Any]], manifest: dict[str, Any]) -> int:
    valid = [env for env in envelopes if not is_invalid_detail(env['response'], target)]
    provenance_map = manifest['provenance'][target]
    groups: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for env in valid:
        provenance = provenance_map[env['source_id']]
        if target == 'eflaw':
            payload = detail_payload(env['response'], target) or {}
            basic = payload.get('기본정보', {}) if isinstance(payload.get('기본정보'), dict) else payload
            groups[safe_name(str(basic.get('법령명_한글') or basic.get('법령명한글') or env['source_id']))].append(env)
        else:
            category = (provenance.get('categories') or ['기타'])[0]
            groups[safe_name(category)].append(env)
    for group, rows in groups.items():
        for part_index in range(0, len(rows), MARKDOWN_MAX_DOCUMENTS):
            batch = rows[part_index:part_index + MARKDOWN_MAX_DOCUMENTS]
            suffix = f'.part{part_index // MARKDOWN_MAX_DOCUMENTS + 1:02d}' if len(rows) > MARKDOWN_MAX_DOCUMENTS else ''
            path = RAW_MD_ROOT / target / f'{group}{suffix}.md'
            if path.exists():
                continue
            path.parent.mkdir(parents=True, exist_ok=True)
            temporary_path = path.with_suffix(path.suffix + '.tmp')
            temporary_path.write_text('\n---\n\n'.join(raw_markdown(target, row) for row in batch), encoding='utf-8')
            temporary_path.replace(path)
    return sum(path.read_text(encoding='utf-8').count('<!-- source_id:') for path in (RAW_MD_ROOT / target).glob('*.md'))

## Manifest, 검증, 실행

In [9]:
def initial_manifest() -> dict[str, Any]:
    if MANIFEST_PATH.exists():
        return json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
    return {
        'run_id': RUN_ID, 'run_kind': RUN_KIND, 'sample_mode': SAMPLE_MODE, 'created_at': utc_now(),
        'status': 'running', 'targets': ['eflaw', 'prec', 'expc'],
        'precedent_policy': {'selection_policy': 'latest_per_query', 'search_scope': 'body', 'sort': PRECEDENT_SORT, 'max_per_query': PRECEDENT_MAX_PER_QUERY, 'relevance_evaluated_after_collection': True},
        'request_policy': {'timeout_seconds': REQUEST_TIMEOUT, 'max_attempts': REQUEST_MAX_ATTEMPTS, 'request_interval_seconds': REQUEST_INTERVAL_SECONDS},
        'query_stats': {}, 'detail_counts': {}, 'invalid_details': {}, 'provenance': {},
        'errors': [], 'request_failures': [], 'outputs': {}, 'validation': {},
    }


def save_manifest(manifest: dict[str, Any]) -> None:
    MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')


def file_inventory() -> list[dict[str, Any]]:
    files = []
    for root in (RAW_ROOT, RAW_MD_ROOT, DOCUMENT_ROOT):
        if not root.exists():
            continue
        for path in sorted(item for item in root.rglob('*') if item.is_file() and item != MANIFEST_PATH):
            data = path.read_bytes()
            row = {'path': str(path.relative_to(PROJECT_ROOT)).replace('\\', '/'), 'bytes': len(data), 'sha256': sha256_bytes(data)}
            if path.suffix == '.jsonl':
                row['records'] = sum(1 for line in data.splitlines() if line.strip())
            files.append(row)
    return files


def validate_outputs(manifest: dict[str, Any]) -> dict[str, Any]:
    inventory = file_inventory()
    output_paths = [PROJECT_ROOT / row['path'] for row in inventory]
    oc_files = [str(path.relative_to(PROJECT_ROOT)) for path in output_paths if ('OC' + '=').lower() in path.read_text(encoding='utf-8', errors='ignore').lower()]
    invalid_jsonl: list[str] = []
    record_ids: dict[str, list[str]] = defaultdict(list)
    all_document_ids: list[str] = []
    forbidden_fields = {'chunk_id', 'chunk_index', 'overlap'}
    forbidden_hits: list[str] = []
    for path in (path for path in output_paths if path.suffix == '.jsonl'):
        try:
            with path.open(encoding='utf-8') as handle:
                for line in handle:
                    if not line.strip():
                        continue
                    row = json.loads(line)
                    metadata = row.get('metadata', {})
                    if metadata.get('record_id'):
                        record_ids[str(path)].append(metadata['record_id'])
                        if DOCUMENT_ROOT in path.parents:
                            all_document_ids.append(metadata['record_id'])
                    if forbidden_fields.intersection(metadata):
                        forbidden_hits.append(str(path))
        except (OSError, json.JSONDecodeError):
            invalid_jsonl.append(str(path))
    duplicate_record_files = [path for path, ids in record_ids.items() if len(ids) != len(set(ids))]
    duplicate_record_ids_target_wide = len(all_document_ids) != len(set(all_document_ids))

    def jsonl_record_count(paths: Iterable[Path]) -> int:
        return sum(1 for path in paths for line in path.read_text(encoding='utf-8').splitlines() if line.strip())

    detail_count_mismatches: dict[str, dict[str, int]] = {}
    search_count_mismatches: dict[str, dict[str, int]] = {}
    markdown_count_mismatches: dict[str, dict[str, int]] = {}
    for target in ('eflaw', 'prec', 'expc'):
        actual_detail = jsonl_record_count((RAW_ROOT / target).glob(f'{target}_detail*.jsonl'))
        expected_detail = int(manifest.get('detail_counts', {}).get(target, {}).get('stored', 0))
        if actual_detail != expected_detail:
            detail_count_mismatches[target] = {'expected': expected_detail, 'actual': actual_detail}
        actual_search = jsonl_record_count((RAW_ROOT / target).glob(f'{target}_search*.jsonl'))
        expected_search = sum(int(row.get('page_count', 0)) for row in manifest.get('query_stats', {}).get(target, []))
        if actual_search != expected_search:
            search_count_mismatches[target] = {'expected': expected_search, 'actual': actual_search}
        actual_markdown = sum(path.read_text(encoding='utf-8').count('<!-- source_id:') for path in (RAW_MD_ROOT / target).glob('*.md'))
        expected_markdown = int(manifest.get('detail_counts', {}).get(target, {}).get('valid', 0))
        if actual_markdown != expected_markdown:
            markdown_count_mismatches[target] = {'expected': expected_markdown, 'actual': actual_markdown}
    validation = {
        'checked_at': utc_now(), 'oc_assignment_files': oc_files, 'invalid_jsonl': invalid_jsonl,
        'duplicate_record_id_files': duplicate_record_files, 'duplicate_record_ids_target_wide': duplicate_record_ids_target_wide,
        'detail_count_mismatches': detail_count_mismatches, 'search_count_mismatches': search_count_mismatches,
        'markdown_count_mismatches': markdown_count_mismatches,
        'forbidden_chunk_metadata_files': sorted(set(forbidden_hits)),
        'passed': not (oc_files or invalid_jsonl or duplicate_record_files or duplicate_record_ids_target_wide or forbidden_hits or detail_count_mismatches or search_count_mismatches or markdown_count_mismatches),
    }
    if not validation['passed']:
        raise AssertionError(f'산출물 검증 실패: {validation}')
    return validation


def run_collection() -> dict[str, Any]:
    client = LawApiClient(LAW_API_OC)
    manifest = initial_manifest()
    save_manifest(manifest)
    query_sets = {
        'eflaw': [(query, '법령') for query in EFLAW_QUERIES],
        'prec': PREC_QUERY_CONFIG, 'expc': EXPC_QUERY_CONFIG,
    }
    try:
        for target in ('eflaw', 'prec', 'expc'):
            candidates = collect_candidates(client, target, query_sets[target], manifest)
            envelopes = collect_details(client, target, candidates, manifest)
            manifest['outputs'][target] = {
                'raw_markdown_documents': write_raw_markdown(target, envelopes, manifest),
                'document_records': write_documents(target, envelopes, manifest),
            }
            manifest['request_failures'] = client.failures
            save_manifest(manifest)
        manifest['files'] = file_inventory()
        manifest['validation'] = validate_outputs(manifest)
        manifest['status'] = 'completed'
        manifest['completed_at'] = utc_now()
        save_manifest(manifest)
        return manifest
    except Exception as exc:
        manifest['status'] = 'failed'
        manifest['request_failures'] = client.failures
        manifest['errors'].append({'at': utc_now(), 'type': type(exc).__name__, 'message': str(exc)})
        save_manifest(manifest)
        raise


# Gate 2 승인 후 이 셀을 실행한다. 기본 설정은 소량 sample run이다.
collection_manifest = run_collection()
print(json.dumps({
    'run_id': collection_manifest['run_id'], 'status': collection_manifest['status'],
    'detail_counts': collection_manifest['detail_counts'],
    'outputs': collection_manifest['outputs'], 'validation': collection_manifest['validation'],
}, ensure_ascii=False, indent=2))

[eflaw] 주택임대차보호법: 검색 2 / 수집 1


[eflaw] 주택임대차보호법 시행령: 검색 1 / 수집 1


[eflaw] 민법: 검색 6 / 수집 1


[eflaw] 부동산등기법: 검색 2 / 수집 1


[eflaw] 공인중개사법: 검색 3 / 수집 1


...

[expc] 상세 280/322 | 오류·빈 본문 1


[expc] 상세 300/322 | 오류·빈 본문 1


[expc] 상세 320/322 | 오류·빈 본문 1


[expc] 상세 322/322 | 오류·빈 본문 1


{
  "run_id": "full_20260719_163719",
  "status": "completed",
  "detail_counts": {
    "eflaw": {
      "requested": 16,
      "stored": 16,
      "valid": 16,
      "invalid": 0,
      "invalid_ratio": 0.0
    },
    "prec": {
      "requested": 3752,
      "stored": 3752,
      "valid": 2127,
      "invalid": 1625,
      "invalid_ratio": 0.43310234541577824
    },
    "expc": {
      "requested": 322,
      "stored": 322,
      "valid": 321,
      "invalid": 1,
      "invalid_ratio": 0.003105590062111801
    }
  },
  "outputs": {
    "eflaw": {
      "raw_markdown_documents": 16,
      "document_records": 3146
    },
    "prec": {
      "raw_markdown_documents": 2127,
      "document_records": 4311
    },
    "expc": {
      "raw_markdown_documents": 321,
      "document_records": 962
    }
  },
  "validation": {
    "checked_at": "2026-07-19T08:43:16.881811+00:00",
    "oc_assignment_files": [],
    "invalid_jsonl": [],
    "duplicate_record_id_files": [],
    "duplicate_record_ids